Trước khi đến với giai đoạn xử lý dữ liệu ta sẽ phải kiểm tra chất lượng của dữ liệu

In [50]:
import pandas as pd

raw_data=pd.read_csv('Data/2020-Feb.csv')
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4156682 entries, 0 to 4156681
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     str    
 1   event_type     str    
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  str    
 5   brand          str    
 6   price          float64
 7   user_id        int64  
 8   user_session   str    
dtypes: float64(1), int64(3), str(5)
memory usage: 562.6 MB


In [51]:
raw_data.isna().sum()/3533285*100

event_time         0.000000
event_type         0.000000
product_id         0.000000
category_id        0.000000
category_code    115.459042
brand             51.677348
price              0.000000
user_id            0.000000
user_session       0.029859
dtype: float64

In [52]:
print(raw_data.duplicated().sum())

240290


In [53]:
183860/3533285*100

5.20365608774837

In [54]:
raw_data.describe()

,product_id,category_id,price,user_id
count,4.156682e+06,4.156682e+06,4.156682e+06,4.156682e+06
mean,5.504270e+06,1.562416e+18,8.525847e+00,5.450008e+08
std,1.281809e+06,1.797695e+17,1.958040e+01,8.955674e+07
min,3.752000e+03,1.487580e+18,-7.937000e+01,2.038666e+06
25%,5.733046e+06,1.487580e+18,2.050000e+00,5.084226e+08
50%,5.814703e+06,1.487580e+18,4.110000e+00,5.797951e+08
75%,5.864791e+06,1.487580e+18,6.980000e+00,6.104701e+08
max,5.932595e+06,2.242903e+18,3.277800e+02,6.220902e+08


Xử lý trùng lặp và ngoại lệ price

In [55]:
data = raw_data.drop_duplicates()
data=data[data['price']>=0]

Xử lý ngoại lệ 

In [56]:
gioi_han_99 = data['price'].quantile(0.99)
print(f"Giá trị bách phân vị 99: {gioi_han_99}")

data['price'] = data['price'].clip(upper=gioi_han_99)

data.describe()

Giá trị bách phân vị 99: 97.87


,product_id,category_id,price,user_id
count,3.916358e+06,3.916358e+06,3.916358e+06,3.916358e+06
mean,5.501722e+06,1.562985e+18,8.104267e+00,5.459506e+08
std,1.287350e+06,1.805280e+17,1.443746e+01,8.974873e+07
min,3.752000e+03,1.487580e+18,0.000000e+00,2.038666e+06
25%,5.733052e+06,1.487580e+18,2.060000e+00,5.108183e+08
50%,5.814735e+06,1.487580e+18,4.250000e+00,5.822492e+08
75%,5.864841e+06,1.487580e+18,7.140000e+00,6.107081e+08
max,5.932595e+06,2.242903e+18,9.787000e+01,6.220902e+08


In [57]:
data['brand'] = data['brand'].fillna('Unknown')
data.drop(columns='category_code',inplace=True)
data.drop(columns='user_id',inplace=True)
data=data.dropna()

In [58]:
data.info()

<class 'pandas.DataFrame'>
Index: 3915452 entries, 0 to 4156681
Data columns (total 7 columns):
 #   Column        Dtype  
---  ------        -----  
 0   event_time    str    
 1   event_type    str    
 2   product_id    int64  
 3   category_id   int64  
 4   brand         str    
 5   price         float64
 6   user_session  str    
dtypes: float64(1), int64(2), str(4)
memory usage: 508.6 MB


In [59]:
data.isna().sum()/3533285*100

event_time      0.0
event_type      0.0
product_id      0.0
category_id     0.0
brand           0.0
price           0.0
user_session    0.0
dtype: float64

In [60]:
import numpy as np

data['event_time'] = pd.to_datetime(data['event_time'])
purchased_sessions = data[data['event_type'] == 'purchase']['user_session'].unique()

data_features = data[data['event_type'] != 'purchase'].copy()

event_counts = data_features.groupby(['user_session', 'event_type']).size().unstack(fill_value=0).reset_index()

for col in ['view', 'cart', 'remove_from_cart']:
    if col not in event_counts.columns:
        event_counts[col] = 0
event_counts = event_counts.rename(columns={
    'view': 'total_views', 
    'cart': 'total_carts', 
    'remove_from_cart': 'total_removes'
})

session_stats = data_features.groupby('user_session').agg(
    # Thời lượng phiên truy cập (Giây)
    session_duration=('event_time', lambda x: (x.max() - x.min()).total_seconds()),
    
    # Giá trung bình và Giá cao nhất của các sản phẩm họ đã xem/bỏ giỏ
    avg_price=('price', 'mean'),
    max_price=('price', 'max'),
    
    # Số lượng sản phẩm khác nhau và thương hiệu khác nhau họ đã tương tác
    unique_products=('product_id', 'nunique'),
    unique_brands=('brand', 'nunique')
).reset_index()

df_train = pd.merge(event_counts, session_stats, on='user_session')

# Gắn nhãn (1: Có mua hàng, 0: Không mua)
df_train['is_purchased'] = df_train['user_session'].isin(purchased_sessions).astype(int)

# Điền 0 nếu có giá trị null phát sinh trong lúc gộp
df_train = df_train.fillna(0)

print(df_train.head())

                           user_session  total_carts  total_removes  \
0  000090e1-da13-42b1-a31b-91a9ee5e6a88            0              0   
1  0000a4ee-e357-459f-975f-6f8b30f1bdb8            0              0   
2  0000b84a-f3de-4ab6-84c7-ddb6e410f64d            0              0   
3  0000d04d-3516-42ea-bdb9-584e92ce619e            6             11   
4  0000de26-bd58-42c9-9173-4763c76b398e            0              0   

   total_views  session_duration  avg_price  max_price  unique_products  \
0            1               0.0   6.370000       6.37                1   
1           42             335.0   5.972619      12.98               37   
2            1               0.0   3.000000       3.00                1   
3            1             173.0   1.695000      10.40               10   
4            1               0.0   2.220000       2.22                1   

   unique_brands  is_purchased  
0              1             0  
1              2             0  
2              1       

In [61]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 928385 entries, 0 to 928384
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_session      928385 non-null  str    
 1   total_carts       928385 non-null  int64  
 2   total_removes     928385 non-null  int64  
 3   total_views       928385 non-null  int64  
 4   session_duration  928385 non-null  float64
 5   avg_price         928385 non-null  float64
 6   max_price         928385 non-null  float64
 7   unique_products   928385 non-null  int64  
 8   unique_brands     928385 non-null  int64  
 9   is_purchased      928385 non-null  int64  
dtypes: float64(3), int64(6), str(1)
memory usage: 102.8 MB


In [62]:
df_train.to_csv('Data/Train4.csv',index=False)